# Topic 20: Training a CNN: MNIST, Fashion-MNIST, and the MLP Compared

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 20 lecture note](https://dongchul.kim/csci3320-fall2026/topic/22), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## Part 1: the data

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
train = torchvision.datasets.MNIST("data", train=True, download=True)
test = torchvision.datasets.MNIST("data", train=False, download=True)
X_train = train.data.unsqueeze(1).float() / 255     # (60000, 1, 28, 28): keep the grid
X_test = test.data.unsqueeze(1).float() / 255       # (10000, 1, 28, 28)
loader = DataLoader(TensorDataset(X_train, train.targets), batch_size=128, shuffle=True)

print(train.data.shape, "->", X_train.shape)
print("one batch:", next(iter(loader))[0].shape, " batches per epoch:", len(loader))

## Part 2: the model, the loss, the optimizer

In [ ]:
torch.manual_seed(0)
loader = DataLoader(TensorDataset(X_train, train.targets), batch_size=128, shuffle=True)
model = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("weights:", sum(p.numel() for p in model.parameters()))

## Part 3: training

In [ ]:
for epoch in range(10):
    for xb, yb in loader:
        optimizer.zero_grad()          # 1. clear old gradients
        pred = model(xb)               # 2. forward: make predictions
        loss = loss_fn(pred, yb)       # 3. measure how wrong
        loss.backward()                # 4. backward: find every gradient
        optimizer.step()               # 5. step: nudge every weight
    print(f"epoch {epoch + 1}   loss {loss.item():.4f}")

## Part 4: testing

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
acc = (pred == test.targets).float().mean()
print(f"test accuracy: {acc.item():.4f}")

## CNN against MLP on MNIST

In [ ]:
def train_and_test(model, X_train, y_train, X_test, y_test, epochs=10):
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        train_pred = model(X_train).argmax(dim=1)
        test_pred = model(X_test).argmax(dim=1)
    train_acc = (train_pred == y_train).float().mean().item()
    test_acc = (test_pred == y_test).float().mean().item()
    return train_acc, test_acc, test_pred

torch.manual_seed(0)
mlp = nn.Sequential(
    nn.Flatten(),                      # the MLP takes one long row
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10),
)
train_acc, test_acc, _ = train_and_test(mlp, X_train, train.targets, X_test, test.targets)
print("MLP weights:", sum(p.numel() for p in mlp.parameters()))
print(f"MLP train accuracy {train_acc:.4f}, test accuracy {test_acc:.4f}")

## Look inside: the filters it learned

In [ ]:
import matplotlib.pyplot as plt

print(model[0].weight.shape)           # 8 filters, each 1 x 3 x 3
x = X_test[:1]                         # the first test image
print("this digit is a", test.targets[0].item())
with torch.no_grad():
    maps = model[1](model[0](x))       # first Conv2d, then its ReLU
print(maps.shape)                      # 1 image, 8 maps, 28 x 28

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(model[0].weight[i, 0].detach(), cmap="RdBu_r")
    axes[1, i].imshow(maps[0, i], cmap="Blues")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
plt.show()

## A harder problem: Fashion-MNIST

In [ ]:
train = torchvision.datasets.FashionMNIST("data", train=True, download=True)
test = torchvision.datasets.FashionMNIST("data", train=False, download=True)
X_train = train.data.unsqueeze(1).float() / 255
X_test = test.data.unsqueeze(1).float() / 255
print(train.data.shape, test.data.shape)
print(test.classes)

torch.manual_seed(0)
model = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10),
)
train_acc, test_acc, pred = train_and_test(model, X_train, train.targets, X_test, test.targets)
print(f"CNN train accuracy {train_acc:.4f}, test accuracy {test_acc:.4f}")

## Which classes does it confuse?

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test.targets, pred)
print(cm)

for i, name in enumerate(test.classes):
    print(f"{name:12s} {cm[i, i] / cm[i].sum():.3f} of its images are correct")

## The error you will see: forgetting unsqueeze

In [ ]:
try:
    model(test.data[:128].float() / 255)      # no unsqueeze(1)
except RuntimeError as e:
    print("RuntimeError:", e)

print(model(test.data[:128].unsqueeze(1).float() / 255).shape)

## Your turn: a wider CNN, by hand

In [ ]:
wider = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
)
x = torch.zeros(1, 1, 28, 28)
for layer in wider:
    x = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    print(f"{type(layer).__name__:10s} {str(tuple(x.shape)):18s} {n:6d} weights")
print("total:", sum(p.numel() for p in wider.parameters()))

Back to the [Topic 20 lecture note](https://dongchul.kim/csci3320-fall2026/topic/22).